In [11]:
import pandas as pd
import matplotlib.pyplot as plt

In [12]:
df = pd.read_csv("../data/raw/prices_DK2_2024_08_10_2025_10_01.csv")

df["HourUTC"] = pd.to_datetime(df["HourUTC"])
df["HourDK"] = pd.to_datetime(df["HourDK"])

df = df.sort_values("HourUTC").reset_index(drop=True)

df.head()

,HourUTC,HourDK,PriceArea,SpotPriceDKK,SpotPriceEUR
0,2024-08-10 06:00:00,2024-08-10 08:00:00,DK2,112.010002,15.01
1,2024-08-10 07:00:00,2024-08-10 09:00:00,DK2,1.490000,0.20
2,2024-08-10 08:00:00,2024-08-10 10:00:00,DK2,-22.389999,-3.00
3,2024-08-10 09:00:00,2024-08-10 11:00:00,DK2,-91.040001,-12.20
4,2024-08-10 10:00:00,2024-08-10 12:00:00,DK2,-167.830002,-22.49


In [13]:
df = df[["HourUTC","HourDK", "PriceArea", "SpotPriceEUR"]].copy()

df.head()

,HourUTC,HourDK,PriceArea,SpotPriceEUR
0,2024-08-10 06:00:00,2024-08-10 08:00:00,DK2,15.01
1,2024-08-10 07:00:00,2024-08-10 09:00:00,DK2,0.20
2,2024-08-10 08:00:00,2024-08-10 10:00:00,DK2,-3.00
3,2024-08-10 09:00:00,2024-08-10 11:00:00,DK2,-12.20
4,2024-08-10 10:00:00,2024-08-10 12:00:00,DK2,-22.49


In [14]:

df["hour"] = df["HourDK"].dt.hour
df["day_of_week"] = df["HourDK"].dt.dayofweek
df["month"] = df["HourDK"].dt.month
df["year"] = df["HourDK"].dt.year
df["is_weekend"] = df["HourDK"].dt.dayofweek.isin([5,6]).astype(int)

In [15]:
df["price_lag_24h"] = df["SpotPriceEUR"].shift(24)
df["price_lag_48h"] = df["SpotPriceEUR"].shift(48)
df["price_lag_168h"] = df["SpotPriceEUR"].shift(168)

In [16]:
df["price_rolling_mean_24h"] = df["SpotPriceEUR"].shift(1).rolling(window=24).mean()
df["price_rolling_std_24h"] = df["SpotPriceEUR"].shift(1).rolling(window=24).std()

df["price_rolling_mean_168h"] = df["SpotPriceEUR"].shift(1).rolling(window=168).mean()
df["price_rolling_std_168h"] = df["SpotPriceEUR"].shift(1).rolling(window=168).std()

In [17]:
df["target_next_hour"] = df["SpotPriceEUR"].shift(-1)

In [18]:
df_model = df.dropna().reset_index(drop=True)

df_model.head()

,HourUTC,HourDK,PriceArea,SpotPriceEUR,hour,day_of_week,month,year,is_weekend,price_lag_24h,price_lag_48h,price_lag_168h,price_rolling_mean_24h,price_rolling_std_24h,price_rolling_mean_168h,price_rolling_std_168h,target_next_hour
0,2024-08-17 06:00:00,2024-08-17 08:00:00,DK2,103.430000,8,5,8,2024,1,124.830002,114.970001,15.01,109.835416,49.468326,77.672202,57.698791,84.000000
1,2024-08-17 07:00:00,2024-08-17 09:00:00,DK2,84.000000,9,5,8,2024,1,100.949997,91.120003,0.20,108.943749,49.379084,78.198512,57.526797,69.489998
2,2024-08-17 08:00:00,2024-08-17 10:00:00,DK2,69.489998,10,5,8,2024,1,76.150002,59.099998,-3.00,108.237499,49.619018,78.697321,57.208862,54.160000
3,2024-08-17 09:00:00,2024-08-17 11:00:00,DK2,54.160000,11,5,8,2024,1,59.830002,19.940001,-12.20,107.959999,49.824472,79.128810,56.861300,46.830002
4,2024-08-17 10:00:00,2024-08-17 12:00:00,DK2,46.830002,12,5,8,2024,1,50.680000,9.860000,-22.49,107.723749,50.075420,79.523810,56.452084,40.630001


In [19]:
df_model.to_csv("../data/processed/dk2_price_features_next_hour.csv", index=False)

In [20]:
print("Rows:", len(df_model))
print("Columns:", df_model.columns.tolist())
print(df_model.isna().sum())

Rows: 9831
Columns: ['HourUTC', 'HourDK', 'PriceArea', 'SpotPriceEUR', 'hour', 'day_of_week', 'month', 'year', 'is_weekend', 'price_lag_24h', 'price_lag_48h', 'price_lag_168h', 'price_rolling_mean_24h', 'price_rolling_std_24h', 'price_rolling_mean_168h', 'price_rolling_std_168h', 'target_next_hour']
HourUTC                    0
HourDK                     0
PriceArea                  0
SpotPriceEUR               0
hour                       0
day_of_week                0
month                      0
year                       0
is_weekend                 0
price_lag_24h              0
price_lag_48h              0
price_lag_168h             0
price_rolling_mean_24h     0
price_rolling_std_24h      0
price_rolling_mean_168h    0
price_rolling_std_168h     0
target_next_hour           0
dtype: int64
